In [1]:
! pip install mlxtend


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ast
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# 1. Load data
df_basket = pd.read_csv('df_basket.csv')
df_basket

,Order ID,Products,n_items
0,AE-2011-9160,"['Storage', 'Machines']",2
1,AE-2013-1130,"['Bookcases', 'Fasteners']",2
2,AE-2013-1530,"['Storage', 'Supplies']",2
3,AE-2014-3830,"['Phones', 'Storage', 'Art', 'Storage', 'Binde...",6
4,AG-2011-1390,"['Paper', 'Binders']",2
...,...,...,...
12773,ZI-2014-5970,"['Chairs', 'Accessories', 'Labels']",3
12774,ZI-2014-6740,"['Appliances', 'Supplies', 'Art', 'Storage', '...",6
12775,ZI-2014-7160,"['Storage', 'Storage', 'Paper']",3
12776,ZI-2014-7610,"['Machines', 'Fasteners']",2


In [3]:
# 2. Konversi kolom Products: string list → list Python
df_basket['Products'] = df_basket['Products'].apply(ast.literal_eval)

# 3. TransactionEncoder → format boolean matrix
te = TransactionEncoder()
te_array = te.fit_transform(df_basket['Products'])
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

In [4]:
frequent_itemsets = apriori(df_encoded, min_support=0.005, use_colnames=True) 
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.01)

rules_filtered = rules[
    (rules['confidence'] >= 0.1) &
    (rules['lift'] >= 1.0)
].sort_values('lift', ascending=False)

# Ambil kolom yang penting saja
rules_filtered = rules_filtered[
    [
        'antecedents',
        'consequents',
        'support',
        'confidence',
        'lift'
    ]
]

print(f"Frequent itemsets: {len(frequent_itemsets)}")
print(f"Total rules: {len(rules)}")
print(f"Rules setelah filter: {len(rules_filtered)}")

rules_filtered.head(10)

Frequent itemsets: 271
Total rules: 1000
Rules setelah filter: 108


,antecedents,consequents,support,confidence,lift
426,"frozenset({Binders, Paper})",frozenset({Appliances}),0.007278,0.135965,1.349930
412,"frozenset({Storage, Appliances})",frozenset({Art}),0.006965,0.327206,1.321023
623,"frozenset({Storage, Machines})",frozenset({Art}),0.006809,0.323420,1.305738
424,"frozenset({Appliances, Binders})",frozenset({Paper}),0.007278,0.237852,1.277540
624,"frozenset({Art, Machines})",frozenset({Storage}),0.006809,0.325843,1.262085
436,"frozenset({Storage, Appliances})",frozenset({Binders}),0.008139,0.382353,1.229418
622,"frozenset({Storage, Art})",frozenset({Machines}),0.006809,0.104442,1.227743
425,"frozenset({Appliances, Paper})",frozenset({Binders}),0.007278,0.367589,1.181945
587,"frozenset({Storage, Fasteners})",frozenset({Art}),0.008765,0.292428,1.180615
414,"frozenset({Appliances, Art})",frozenset({Storage}),0.006965,0.301695,1.168553


### Interpretasi Hasil Association Rules

Berdasarkan penerapan algoritma Apriori pada data transaksi, diperoleh 271 frequent itemsets dan 1000 association rules. Setelah dilakukan penyaringan menggunakan nilai confidence minimal 0,1 dan lift minimal 1, diperoleh 108 aturan asosiasi yang memenuhi kriteria. Berikut adalah interpretasi dari beberapa aturan dengan nilai lift tertinggi.

#### 1. Rule: {Paper, Binders} → {Appliances}

Aturan ini memiliki nilai support sebesar 0,73%, confidence sebesar 13,60%, dan lift sebesar 1,35. Nilai support menunjukkan bahwa kombinasi produk Paper, Binders, dan Appliances muncul pada sekitar 0,73% dari seluruh transaksi. Nilai confidence sebesar 13,60% menunjukkan bahwa pelanggan yang membeli Paper dan Binders memiliki peluang sebesar 13,60% untuk juga membeli Appliances. Sementara itu, nilai lift sebesar 1,35 menunjukkan bahwa pembelian Appliances terjadi 1,35 kali lebih sering dibandingkan kondisi normal ketika pelanggan membeli Paper dan Binders.

Hasil ini menunjukkan adanya hubungan positif antara produk Paper, Binders, dan Appliances. Oleh karena itu, perusahaan dapat mempertimbangkan strategi penempatan produk yang berdekatan atau menawarkan paket promosi untuk meningkatkan penjualan silang.

#### 2. Rule: {Appliances, Storage} → {Art}

Aturan ini memiliki support sebesar 0,70%, confidence sebesar 32,72%, dan lift sebesar 1,32. Nilai confidence yang cukup tinggi menunjukkan bahwa sekitar 32,72% pelanggan yang membeli Appliances dan Storage juga membeli produk Art. Nilai lift yang lebih besar dari 1 menunjukkan adanya hubungan positif antara ketiga kategori produk tersebut.

Temuan ini menunjukkan bahwa pelanggan yang membeli perlengkapan penyimpanan dan peralatan tertentu juga memiliki kecenderungan membeli produk kategori Art. Informasi ini dapat dimanfaatkan untuk memberikan rekomendasi produk tambahan kepada pelanggan.

#### 3. Rule: {Machines, Storage} → {Art}

Aturan ini memiliki support sebesar 0,68%, confidence sebesar 32,34%, dan lift sebesar 1,31. Hal ini menunjukkan bahwa sekitar 32,34% pelanggan yang membeli Machines dan Storage juga membeli Art. Nilai lift sebesar 1,31 menunjukkan bahwa hubungan tersebut lebih kuat dibandingkan pembelian secara acak.

Perusahaan dapat memanfaatkan pola ini untuk menerapkan strategi cross-selling dengan menawarkan produk Art kepada pelanggan yang membeli Machines dan Storage.

#### 4. Rule: {Appliances, Binders} → {Paper}

Aturan ini memiliki support sebesar 0,73%, confidence sebesar 23,79%, dan lift sebesar 1,28. Artinya, pelanggan yang membeli Appliances dan Binders memiliki peluang sebesar 23,79% untuk membeli Paper. Nilai lift sebesar 1,28 menunjukkan adanya hubungan positif antara produk-produk tersebut.

Hasil ini mengindikasikan bahwa produk Paper sering dibeli bersama dengan Appliances dan Binders. Oleh karena itu, perusahaan dapat mempertimbangkan penyusunan paket penjualan atau penempatan produk yang saling berdekatan.

#### 5. Rule: {Machines, Art} → {Storage}

Aturan ini memiliki support sebesar 0,68%, confidence sebesar 32,58%, dan lift sebesar 1,26. Nilai confidence menunjukkan bahwa sekitar sepertiga pelanggan yang membeli Machines dan Art juga membeli Storage. Nilai lift yang lebih besar dari 1 mengindikasikan adanya hubungan positif antar produk.

Temuan ini dapat dimanfaatkan untuk meningkatkan efektivitas promosi dengan menawarkan produk Storage kepada pelanggan yang membeli Machines dan Art.

#### 6. Rule: {Appliances, Storage} → {Binders}

Aturan ini memiliki support sebesar 0,81%, confidence sebesar 38,24%, dan lift sebesar 1,23. Nilai confidence yang mendekati 40% menunjukkan bahwa hubungan antar produk cukup kuat. Nilai lift sebesar 1,23 juga menunjukkan bahwa pelanggan yang membeli Appliances dan Storage lebih cenderung membeli Binders dibandingkan pelanggan secara umum.

Strategi yang dapat diterapkan adalah memberikan rekomendasi Binders kepada pelanggan yang membeli Appliances dan Storage.

#### 7. Rule: {Storage, Art} → {Machines}

Aturan ini memiliki support sebesar 0,68%, confidence sebesar 10,44%, dan lift sebesar 1,23. Meskipun nilai confidence relatif lebih rendah dibandingkan aturan lainnya, nilai lift yang masih lebih besar dari 1 menunjukkan adanya hubungan positif antara produk tersebut.

Perusahaan dapat mempertimbangkan promosi tambahan untuk meningkatkan pembelian Machines pada pelanggan yang membeli Storage dan Art.

#### 8. Rule: {Appliances, Paper} → {Binders}

Aturan ini memiliki support sebesar 0,73%, confidence sebesar 36,76%, dan lift sebesar 1,18. Artinya, lebih dari sepertiga pelanggan yang membeli Appliances dan Paper juga membeli Binders. Nilai lift menunjukkan bahwa hubungan ini lebih kuat dibandingkan pembelian secara acak.

Temuan ini menunjukkan bahwa produk Paper dan Binders memiliki keterkaitan yang cukup erat dalam transaksi pelanggan.

#### 9. Rule: {Fasteners, Storage} → {Art}

Aturan ini memiliki support sebesar 0,88%, confidence sebesar 29,24%, dan lift sebesar 1,18. Nilai confidence menunjukkan bahwa hampir 30% pelanggan yang membeli Fasteners dan Storage juga membeli Art. Nilai lift menunjukkan adanya hubungan positif antar produk.

Pola ini dapat dimanfaatkan dalam strategi promosi atau rekomendasi produk untuk meningkatkan nilai transaksi pelanggan.

#### 10. Rule: {Appliances, Art} → {Storage}

Aturan ini memiliki support sebesar 0,70%, confidence sebesar 30,17%, dan lift sebesar 1,17. Artinya, pelanggan yang membeli Appliances dan Art memiliki peluang sekitar 30% untuk membeli Storage. Nilai lift menunjukkan bahwa hubungan tersebut lebih kuat dibandingkan kondisi normal.

Perusahaan dapat memanfaatkan hasil ini untuk menawarkan produk Storage sebagai produk pelengkap kepada pelanggan yang membeli Appliances dan Art.

### Kesimpulan

Hasil Market Basket Analysis menunjukkan bahwa terdapat hubungan positif antara beberapa kategori produk, terutama yang melibatkan Paper, Binders, Appliances, Storage, Art, Machines, dan Fasteners. Seluruh aturan yang dianalisis memiliki nilai lift lebih besar dari 1, yang menunjukkan adanya kecenderungan produk-produk tersebut dibeli secara bersamaan. Temuan ini dapat dimanfaatkan untuk mendukung strategi cross-selling, bundling produk, penataan rak, serta sistem rekomendasi produk guna meningkatkan penjualan dan nilai transaksi pelanggan.
